# S3-AI03 Hybrid RAG retrieval validation
Runs the locked `v0.1.0` validation set with BGE-M3 embeddings and the BGE reranker on a Kaggle GPU.

In [ ]:
from pathlib import Path
import json
import sys
import torch

assert torch.cuda.is_available(), 'Enable a GPU accelerator in Kaggle before continuing.'
print('Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA devices:', torch.cuda.device_count())

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working/s3-ai03')
WORKING_ROOT.mkdir(parents=True, exist_ok=True)

project_roots = [p.parent for p in INPUT_ROOT.rglob('pyproject.toml') if (p.parent / 'src' / 'biodiversity_ai').is_dir()]
assert project_roots, 'Kaggle input does not contain the prepared S3-AI03 bundle.'
PROJECT_ROOT = project_roots[0]
DATASET_DIR = PROJECT_ROOT / 'datasets' / 'v0.1.0'
EMBEDDING_DIR = PROJECT_ROOT / 'artifacts' / 'rag' / 'v0.1.0' / 'bge-m3'
assert (DATASET_DIR / 'manifest.json').is_file(), DATASET_DIR
assert (EMBEDDING_DIR / 'manifest.json').is_file(), EMBEDDING_DIR
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Project:', PROJECT_ROOT)
print('Dataset:', DATASET_DIR)
print('Embeddings:', EMBEDDING_DIR)

In [ ]:
from biodiversity_ai.assistant.contracts import DatasetPartition
from biodiversity_ai.assistant.rag_workflow import evaluate_retrieval_variants, write_comparison

comparison = evaluate_retrieval_variants(
    DATASET_DIR,
    EMBEDDING_DIR,
    partition=DatasetPartition.VALIDATION,
    limit=5,
    candidate_limit=20,
    example_limit=500,
    batch_size=32,
    device='cuda',
    reranker_model='BAAI/bge-reranker-v2-m3',
)
OUTPUT_PATH = WORKING_ROOT / 'retrieval-validation.json'
write_comparison(OUTPUT_PATH, comparison)
print('Saved:', OUTPUT_PATH)

In [ ]:
result = json.loads(OUTPUT_PATH.read_text(encoding='utf-8'))
print(json.dumps(result, ensure_ascii=False, indent=2))